# Euler implícito: calcular el estado siguiente

_Física Computacional 2026_

> Notebook generado a partir de la clase `01-1-implicitos.qmd`. Ejecuten las celdas en orden.

> **Pregunta de la clase.** Si conocemos el estado de un sistema en $t_n$,
> ¿conviene calcular el siguiente estado con la pendiente actual o exigir que
> la pendiente en $t_{n+1}$ sea compatible con el resultado? ¿Cómo comprobamos
> que lo obtenido representa la solución de las ecuaciones?

Al terminar podremos escribir un paso de Euler implícito, resolver la ecuación
que produce con Newton–Raphson cuando sea necesario, y comparar dos métodos
numéricos con una solución analítica. Usaremos como ejemplo un sistema de dos
especies que se producen y decaen. El oscilador quedará planteado al final como
problema para el [TP 01](../TPs/01-oscilador/index.md).

## 1. Presentación: del modelo a la evidencia

Una simulación requiere una cadena de decisiones:

$$
\text{fenómeno}\longrightarrow\text{modelo}\longrightarrow
\text{método numérico}\longrightarrow\text{código}\longrightarrow
\text{resultado}.
$$

Cuando el resultado parece razonable, todavía podemos preguntar si las
ecuaciones representan el fenómeno, si el método aproxima esas ecuaciones y si
el código implementa el método elegido. En esta clase tendremos una **solución
analítica del modelo** para estudiar las dos últimas preguntas. La comparación
con esa solución no demuestra, por sí sola, que el modelo describa un generador
real en todas las condiciones.

Trabajaremos con un problema de valores iniciales,

$$
\frac{d\mathbf y}{dt}=\mathbf f(t,\mathbf y),
\qquad \mathbf y(t_0)=\mathbf y_0,
$$

y con una grilla $t_n=t_0+nh$, donde $h>0$ es el paso temporal. En
[00-2](00-2-euler.qmd) ya vimos cómo convertir ecuaciones de segundo orden en
sistemas de primer orden, construir Euler explícito y medir su error al refinar
$h$. Aquí retomamos esas herramientas para contrastarlas con un método
implícito.

**Antes de calcular:** para una cantidad que sólo decae, ¿esperamos que una
aproximación numérica pueda volverse negativa? Si un método mantiene los
valores positivos, ¿alcanza eso para considerarlo preciso?

## 2. Repaso de Euler explícito

Euler explícito usa la pendiente conocida al comienzo del paso:

$$
\boxed{\mathbf y_{n+1}=\mathbf y_n+h\,\mathbf f(t_n,\mathbf y_n).}
$$

Para un decaimiento simple $N'=-\lambda N$, con $\lambda>0$,

$$
N_{n+1}=(1-\lambda h)N_n,
\qquad N(t)=N_0e^{-\lambda t}.
$$

El factor $1-\lambda h$ aproxima $e^{-\lambda h}$. Si $\lambda h>1$, el
resultado cambia de signo en un solo paso aunque la solución exacta sea
positiva. Si $\lambda h\geq 2$, su magnitud ya no disminuye paso a paso: para
este problema, Euler explícito no es estable con ese paso. Incluso cuando
$0<\lambda h<1$, la trayectoria puede tener un error importante. **Estabilidad,
positividad y precisión son preguntas distintas.**

En el régimen de pasos suficientemente pequeños, el error global de Euler a
tiempo final fijo es proporcional a $h$. Por eso, al dividir $h$ por dos,
esperamos aproximadamente la mitad del error. Esta predicción se comprobará
en el sistema acoplado.

## 3. Euler implícito y Newton–Raphson

Partimos otra vez de $d\mathbf y/dt=\mathbf f(t,\mathbf y)$. Entre dos
instantes consecutivos, la variación exacta del estado se puede escribir como

$$
\mathbf y(t_{n+1})-\mathbf y(t_n)
=\int_{t_n}^{t_{n+1}}\mathbf f(t,\mathbf y(t))\,dt.
$$

La integral reúne todas las pendientes del intervalo. En Euler explícito la
aproximamos mediante un rectángulo cuya altura es la pendiente al comienzo.
Ahora tomamos la altura en el **extremo derecho**, $t_{n+1}$. Como
$h=t_{n+1}-t_n$,

$$
\int_{t_n}^{t_{n+1}}\mathbf f(t,\mathbf y(t))\,dt
\approx h\,\mathbf f(t_{n+1},\mathbf y(t_{n+1})).
$$

Al reemplazar los valores exactos por las aproximaciones numéricas de cada
extremo, obtenemos la regla de **Euler implícito**:

$$
\boxed{\mathbf y_{n+1}=\mathbf y_n+h\,\mathbf f(t_{n+1},\mathbf y_{n+1}).}
$$

La diferencia con Euler explícito es el punto donde se evalúa la pendiente.
Ahora la incógnita $\mathbf y_{n+1}$ aparece en ambos lados: cada paso exige
resolver una ecuación o, para un estado vectorial, un sistema de ecuaciones.
La regla del rectángulo en el extremo derecho comete un error local de orden
$h^2$ para una solución suficientemente suave; bajo las condiciones usuales,
el error global del método es de orden $h$.

### 3.1. Un caso que podemos despejar

Para $N'=-\lambda N$, Euler implícito da

$$
N_{n+1}=N_n-h\lambda N_{n+1}
\quad\Longrightarrow\quad
\boxed{N_{n+1}=\frac{N_n}{1+\lambda h}.}
$$

Si $N_n>0$, este paso conserva la positividad y reduce $N$ para cualquier
$h>0$. Además, el factor $1/(1+\lambda h)$ siempre tiene módulo menor que
uno. Eso no significa que un paso grande sea preciso: con $\lambda h=2$, el
factor implícito es $1/3$, mientras que el factor exacto es $e^{-2}\simeq
0.135$. Los dos Euler tienen **orden global uno**; su comportamiento frente a
pasos grandes es diferente.

### 3.2. Cuando el paso no se puede despejar fácilmente

Para una ecuación escalar $y'=f(t,y)$, llamemos $z$ al valor desconocido
$y_{n+1}$. Podemos llevar el paso implícito a un problema de **búsqueda de
raíces**:

$$
z=y_n+h f(t_{n+1},z)
\quad\Longleftrightarrow\quad
G(z)\equiv z-y_n-h f(t_{n+1},z)=0.
$$

El *residuo* $G(z)$ mide cuánto le falta a una estimación $z$ para satisfacer
la ecuación del paso. Si podemos despejar $z$, como en el decaimiento lineal,
no hace falta un algoritmo iterativo. En otros casos podemos usar
**Newton–Raphson** para encontrar un cero de $G$.

Partimos de una estimación $z^{(0)}$. Cerca de una estimación cualquiera
$z^{(j)}$, aproximamos la función por su recta tangente:

$$
G(z)\approx G(z^{(j)})+G'(z^{(j)})(z-z^{(j)}).
$$

Elegimos la estimación siguiente donde esa recta cruza el eje horizontal.
Igualar a cero el lado derecho y despejar $z$ da

$$
\boxed{z^{(j+1)}=z^{(j)}-\frac{G(z^{(j)})}{G'(z^{(j)})}}.
$$

Así, cada iteración evalúa el residuo y la pendiente de la tangente en la
estimación actual. Repitiendo el proceso podemos acercarnos a una raíz; cerca
de una raíz simple suele hacerlo rápidamente. Si $G'(z^{(j)})$ es cero, la
fórmula no se puede aplicar, y una mala estimación inicial puede llevar a otra
raíz o impedir la convergencia.

Veámoslo en un paso concreto. Tomemos el modelo no lineal $y'=-k y^2$, con
$k>0$. Euler implícito exige encontrar una raíz de

$$
G(z)=z-y_n+hkz^2=0.
$$

Su derivada es $G'(z)=1+2hkz$. Si $y_n=h=k=1$, tenemos
$G(z)=z^2+z-1$. Tomamos $z^{(0)}=y_n=1$. Como $G(1)=1$ y $G'(1)=3$,
la primera tangente es $1+3(z-1)$; corta el eje en
$z^{(1)}=1-1/3=2/3$. Repetimos desde allí: $G(2/3)=1/9$ y
$G'(2/3)=7/3$, por lo que

$$
z^{(2)}=\frac23-\frac{1/9}{7/3}=\frac{13}{21}\approx0.619048.
$$

La siguiente estimación ya es aproximadamente $0.618034$; la raíz positiva
exacta es $(\sqrt5-1)/2$. El índice $j$ cuenta las **iteraciones internas de
Newton**. Sólo después de resolver $G(z)=0$ aceptamos $z$ como
$y_{n+1}$ y avanzamos el índice temporal $n$.

En un programa detenemos Newton cuando $|G(z)|$ sea menor que una tolerancia
elegida, y declaramos una falla si no lo logra tras un número máximo de
iteraciones. Una variación pequeña entre dos estimaciones no alcanza si el
residuo sigue siendo grande. En el siguiente código imprimimos cada
estimación junto con su residuo para comprobar la resolución del paso.

In [ ]:
def newton(g, dg, z_inicial, tol=1e-12, max_iter=20):
    z = z_inicial
    historial = []
    for j in range(max_iter):
        residuo = g(z)
        historial.append((j, z, residuo))
        if abs(residuo) < tol:
            return z, historial
        derivada = dg(z)
        if abs(derivada) < 1e-14:
            raise ArithmeticError("Newton encontró una derivada casi nula")
        z -= residuo / derivada
    raise ArithmeticError("Newton no alcanzó la tolerancia")


y_n, h, k = 1.0, 1.0, 1.0


def g(z):
    return z - y_n + h * k * z**2


def dg(z):
    return 1 + 2 * h * k * z


y_siguiente, historial = newton(g, dg, z_inicial=y_n)
print("iteración    estimación      residuo")
for j, z, residuo in historial:
    print(f"{j:9d} {z:13.9f} {residuo:12.3e}")
print(f"y_(n+1) = {y_siguiente:.9f}")

**Para discutir:** Newton resuelve una ecuación *dentro de cada paso*; no
convierte automáticamente a Euler implícito en un método de mayor orden. El
error de resolver $G(z)=0$ debe ser suficientemente pequeño como para no
dominar el error de discretización.

## 4. Un sistema acoplado con solución analítica

El $^{99m}\mathrm{Tc}$ se usa en estudios de medicina nuclear, incluidos los
de SPECT. Puede obtenerse de un generador cargado con $^{99}\mathrm{Mo}$.
Según el [OIEA](https://www-pub.iaea.org/MTCD/Publications/PDF/Pub1589_web.pdf),
la semivida del $^{99}\mathrm{Mo}$ es aproximadamente $66$ horas y alrededor
del $88.6\%$ de sus decaimientos produce $^{99m}\mathrm{Tc}$; la semivida de
este último es aproximadamente $6$ horas. Un
[resumen del OIEA sobre diagnóstico por imágenes](https://www.iaea.org/sites/default/files/bull554dec2014.pdf)
describe su uso en SPECT.

Modelamos el intervalo **entre dos extracciones** del tecnecio. Sean $M(t)$ y
$T(t)$ las cantidades de $^{99}\mathrm{Mo}$ y $^{99m}\mathrm{Tc}$, expresadas
como fracciones de la cantidad inicial de molibdeno. Tomamos
$M(0)=1$, $T(0)=0$ y

$$
\lambda_M=\frac{\ln 2}{66\,\mathrm h},
\qquad
\lambda_T=\frac{\ln 2}{6\,\mathrm h},
\qquad b=0.886.
$$

El modelo supone tasas de decaimiento constantes, ninguna extracción durante
el intervalo estudiado y ninguna incorporación posterior de molibdeno. La
rama de decaimiento que no produce $^{99m}\mathrm{Tc}$ queda fuera de nuestras
dos variables. Tampoco modelamos el marcaje químico ni la distribución del
radiofármaco en un paciente.

### 4.1. Ecuaciones y predicción física

$$
\boxed{
\begin{aligned}
M'&=-\lambda_M M,\\
T'&=b\lambda_M M-\lambda_T T.
\end{aligned}}
$$

El primer término de $T'$ lo produce el molibdeno; el segundo representa el
decaimiento del tecnecio. **Antes de ejecutar:** $M$ debe disminuir; $T$
parte de cero, primero aumenta por la producción y más tarde disminuye
cuando se agota la fuente. Ninguna de las dos cantidades puede ser negativa
en este modelo.

En forma matricial, con $\mathbf y=(M,T)^\mathsf T$,

$$
\mathbf y'=A\mathbf y,
\qquad
A=\begin{pmatrix}
-\lambda_M & 0\\
b\lambda_M & -\lambda_T
\end{pmatrix}.
$$

La primera ecuación da $M(t)=e^{-\lambda_Mt}$. Al multiplicar la segunda por
$e^{\lambda_Tt}$, su lado izquierdo se convierte en la derivada de
$e^{\lambda_Tt}T(t)$; integrar de $0$ a $t$ da la referencia analítica

$$
\boxed{
M(t)=e^{-\lambda_Mt},\qquad
T(t)=\frac{b\lambda_M}{\lambda_T-\lambda_M}
\left(e^{-\lambda_Mt}-e^{-\lambda_Tt}\right).}
$$

Se puede comprobar la expresión sin memorizarla: cumple $M(0)=1$, $T(0)=0$
y, al derivarla, satisface las dos ecuaciones. Que $T(t)$ tenga un máximo es
una predicción cuantitativa adicional: el máximo ocurre cuando producción y
decaimiento se igualan, $b\lambda_M M=\lambda_T T$. Con estos parámetros
aparece alrededor de las $23$ horas.

### 4.2. Los dos pasos numéricos

Euler explícito evalúa ambas tasas en el estado conocido:

$$
\begin{aligned}
M_{n+1}&=M_n-h\lambda_M M_n,\\
T_{n+1}&=T_n+h(b\lambda_M M_n-\lambda_T T_n).
\end{aligned}
$$

Euler implícito las evalúa en el estado siguiente. Como este sistema es
**lineal y triangular**, primero podemos despejar $M_{n+1}$ y luego
$T_{n+1}$:

$$
\boxed{
M_{n+1}=\frac{M_n}{1+h\lambda_M},\qquad
T_{n+1}=\frac{T_n+hb\lambda_M M_{n+1}}{1+h\lambda_T}.}
$$

Aquí no necesitamos Newton: resolvimos exactamente el sistema lineal de cada
paso. Si la producción o el decaimiento dependieran de manera no lineal de
$M$ o $T$, aparecería un sistema no lineal de residuos y Newton sería una
posible herramienta para resolverlo.

### 4.3. Experimento de refinamiento

El siguiente código integra hasta un mismo tiempo final, $72$ horas, y
compara el estado $(M,T)$ con la referencia analítica. Las dos componentes
están normalizadas por $M(0)$; por eso podemos usar el mayor de sus errores
absolutos sin mezclar unidades. El código se puede copiar completo en un
notebook o en un archivo Python.

**Predicción previa:** al duplicar el número de pasos, el error de cada Euler
debería acercarse a la mitad en el régimen asintótico. Eso no predice cuál de
los dos tendrá menor error para un paso particular.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

lambda_m = np.log(2) / 66  # 1/h
lambda_t = np.log(2) / 6   # 1/h
rama = 0.886


def exacta(t):
    t = np.asarray(t, dtype=float)
    m = np.exp(-lambda_m * t)
    tc = rama * lambda_m / (lambda_t - lambda_m) * (
        np.exp(-lambda_m * t) - np.exp(-lambda_t * t)
    )
    return np.stack((m, tc), axis=-1)


def integrar(n_pasos, metodo, tiempo_final=72.0):
    tiempos = np.linspace(0.0, tiempo_final, n_pasos + 1)
    h = tiempos[1] - tiempos[0]
    estados = np.empty((n_pasos + 1, 2))
    estados[0] = (1.0, 0.0)

    for n in range(n_pasos):
        m, tc = estados[n]
        if metodo == "explicito":
            m_sig = m - h * lambda_m * m
            tc_sig = tc + h * (rama * lambda_m * m - lambda_t * tc)
        elif metodo == "implicito":
            m_sig = m / (1 + h * lambda_m)
            tc_sig = (tc + h * rama * lambda_m * m_sig) / (1 + h * lambda_t)
        else:
            raise ValueError("método desconocido")
        estados[n + 1] = (m_sig, tc_sig)

    return tiempos, estados


tiempo_final = 72.0
pasos = (6, 12, 24, 48, 96, 192)
referencia = exacta(tiempo_final)

for metodo in ("explicito", "implicito"):
    print(f"\n{metodo}: N   h [h]      error       p observado")
    error_anterior = None
    for n in pasos:
        tiempos, estados = integrar(n, metodo, tiempo_final)
        error = np.max(np.abs(estados[-1] - referencia))
        orden = (
            f"{np.log2(error_anterior / error):.3f}"
            if error_anterior is not None else "—"
        )
        print(f"{n:13d} {tiempos[1] - tiempos[0]:7.3f} "
              f"{error:11.3e} {orden:>12}")
        error_anterior = error

tiempos_ref = np.linspace(0.0, tiempo_final, 400)
plt.plot(tiempos_ref, exacta(tiempos_ref)[:, 1], "k-", label="exacta")
for metodo, estilo in (("explicito", "o--"), ("implicito", "s--")):
    tiempos, estados = integrar(12, metodo, tiempo_final)
    plt.plot(tiempos, estados[:, 1], estilo, label=f"Euler {metodo}")
plt.xlabel("tiempo [h]")
plt.ylabel(r"$T(t)$, fracción de $M(0)$")
plt.legend()
plt.show()

El **orden observado** entre dos filas es

$$
p_{\mathrm{obs}}=
\frac{\log[e(h)/e(h/2)]}{\log 2}.
$$

No basta con ajustar una recta y anunciar su pendiente: hay que mirar si los
últimos cocientes se estabilizan cerca de uno. Tampoco basta con que la figura
se vea bien. La tabla fija el observable, el tiempo final, la referencia y los
pasos usados. Una prueba más exigente podría medir el error en varios tiempos
o preguntar cuánto se desplaza el instante del máximo de $T$.

**Comprobación breve:** antes de interpretar el orden, verificar que la primera
fila de `estados` sea exactamente $(1,0)$, que todos sus elementos sean
finitos y que la referencia analítica satisfaga las condiciones iniciales y
las ecuaciones. Una tolerancia para aceptar una simulación se elegiría según
la cantidad que necesitamos predecir; no se ajusta después sólo para que una
prueba pase.

## 5. El próximo problema: un oscilador

El decaimiento acoplado ofrece una referencia exacta y cantidades que cambian
de forma no periódica. En el [TP 01](../TPs/01-oscilador/index.md) trabajarán
con otro tipo de dinámica: una masa unida a un resorte ideal,

$$
m\frac{d^2x}{dt^2}=-kx,
\qquad
\frac{dx}{dt}=v,\quad \frac{dv}{dt}=-\frac{k}{m}x.
$$

La pregunta inicial será: **¿qué evidencias necesitamos para confiar en una
trayectoria calculada durante muchos ciclos?** Antes de abrir el código o las
figuras del práctico, anoten qué esperan del período, de la amplitud y del
comportamiento a tiempos largos. Después tendrán que decidir qué controles
permiten poner a prueba esas predicciones y qué concluye realmente cada uno.